# Online softmax & tiled attention
**LLMs, Animated · M5.4 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/m5-4-online-softmax.ipynb)

FlashAttention never stores the $n \times n$ attention matrix. It visits the keys **one block at a time** and keeps three running numbers per query row:
a running max $m$, a running sum $\ell$ and an unnormalised output $a$. When a new block $B$ arrives:

$$m' = \max\big(m, \max_{j\in B} s_j\big),\qquad \ell' = e^{m-m'}\ell + \sum_{j\in B} e^{s_j-m'},\qquad a' = e^{m-m'}a + \sum_{j\in B} e^{s_j-m'}v_j$$

and at the end $o = a/\ell$. In this notebook you will:

1. implement that **merge** step and check it on the lesson's worked example
2. run a whole row block by block, including scores that would overflow a naive softmax
3. write a **tiled attention** loop in NumPy and verify it equals standard attention
4. add the **causal mask**, skipping tiles that lie entirely in the future
5. measure how much memory the tiled version saves

NumPy can't show the *speed* benefit (that comes from GPU SRAM vs HBM), but it can prove the algorithm is **exact**. About 20 minutes. Solutions are at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def standard_attention(Q, K, V, causal=False):
    """The reference: builds the full (n, m) score matrix S and probability matrix P."""
    S = Q @ K.T / np.sqrt(Q.shape[1])
    if causal:
        n, m = S.shape
        S = np.where(np.triu(np.ones((n, m), dtype=bool), k=1), -np.inf, S)
    return softmax(S) @ V

## 1. The merge step
Write `merge(state, scores, values)`. The state for an empty row is $m = -\infty$, $\ell = 0$, $a = \mathbf{0}$; NumPy gives $e^{-\infty} = 0$, so the first block needs no special case.

In [ ]:
def merge(state, scores, values):
    """One online-softmax update for a single query row.
    state  = (m, l, a): running max (float), running sum (float), running weighted sum of values (array of shape (d_v,))
    scores = (B,) scores of the new block,  values = (B, d_v) values of the new block.
    Returns the new (m, l, a)."""
    m, l, a = state
    # TODO 1: m_new = the larger of m and the max of `scores`
    # TODO 2: corr = exp(m - m_new)        (the correction factor for the old terms)
    # TODO 3: e = exp(scores - m_new)      (the new block's terms)
    # TODO 4: return m_new, corr*l + sum(e), corr*a + e @ values
    raise NotImplementedError("Fill in merge")

In [ ]:
# The lesson's worked example: scores [2, 1 | 3, 0], values [1, 2 | 3, 4] (as 1-d value vectors)
s = np.array([2., 1., 3., 0.]); v = np.array([[1.], [2.], [3.], [4.]])
st = merge((-np.inf, 0.0, np.zeros(1)), s[:2], v[:2])
assert np.isclose(st[0], 2) and np.isclose(st[1], 1.3679, atol=1e-4) and np.allclose(st[2], [1.7358], atol=1e-4), st
st = merge(st, s[2:], v[2:])
assert np.isclose(st[0], 3) and np.isclose(st[1], 1.5530, atol=1e-4), st
o = st[2] / st[1]
assert np.allclose(o, softmax(s) @ v), (o, softmax(s) @ v)
print("✅ merge works. Worked example output:", o.round(3), "(lesson: 2.471)")

# The faded example: scores [1, 0 | 3, 1], values [2, 4 | 1, 3]
s2 = np.array([1., 0., 3., 1.]); v2 = np.array([[2.], [4.], [1.], [3.]])
st = merge(merge((-np.inf, 0.0, np.zeros(1)), s2[:2], v2[:2]), s2[2:], v2[2:])
print("faded example: l =", round(st[1], 4), " output =", (st[2] / st[1]).round(4), " Compare with your answers in the lesson!")

## 2. A whole row, block by block
Now loop over the keys `block` at a time. The result must not depend on the block size, and it must survive scores like 1000 (where `np.exp(1000)` is `inf`).

In [ ]:
def online_softmax_row(scores, values, block=2):
    """softmax(scores) @ values for ONE query row, visiting the keys `block` at a time."""
    state = (-np.inf, 0.0, np.zeros(values.shape[1]))   # empty state: max = -inf, sum = 0
    # TODO: loop start = 0, block, 2*block, ... and merge in scores[start:start+block], values[start:start+block]
    # TODO: return a / l
    raise NotImplementedError("Fill in online_softmax_row")

In [ ]:
scores = rng.normal(size=37) * 3; values = rng.normal(size=(37, 5))
ref = softmax(scores) @ values
for b in [1, 2, 5, 16, 37, 100]:
    assert np.allclose(online_softmax_row(scores, values, block=b), ref), f"block={b} disagrees"
big = np.array([1000., 1001., 999., 1002.]); vb = np.eye(4)
out = online_softmax_row(big, vb, block=2)
assert np.all(np.isfinite(out)) and np.allclose(out, softmax(big)), out
print("✅ same answer for every block size, and no overflow:", out.round(4))

## 3. Tiled attention
Do the same for a whole **block of query rows** at once: `m` and `l` become vectors (one entry per row) and `acc` a matrix.
The biggest temporary array should be one `(Br, Bc)` tile of scores, never the full `(n, n)` matrix.

In [ ]:
def flash_attention(Q, K, V, Br=16, Bc=16, causal=False):
    """Tiled attention with an online softmax. Never builds an (n, m) matrix:
    the biggest temporary is one (Br, Bc) tile of scores."""
    n, d = Q.shape
    O = np.zeros((n, V.shape[1]))
    scale = 1.0 / np.sqrt(d)
    for i in range(0, n, Br):                       # one block of query rows
        q = Q[i:i + Br]
        rows = np.arange(i, i + len(q))             # absolute positions of these queries (used in exercise 4)
        m = np.full(len(q), -np.inf); l = np.zeros(len(q)); acc = np.zeros((len(q), V.shape[1]))
        for j in range(0, K.shape[0], Bc):          # stream the K/V blocks
            # (exercise 4 adds causal masking here)
            # TODO 1: s = scaled scores for this tile: (q @ K[j:j+Bc].T) * scale      -> shape (len(q), block size)
            # TODO 2: m_new = elementwise max of m and the row-max of s
            # TODO 3: corr = exp(m - m_new);  p = exp(s - m_new[:, None])
            # TODO 4: update l and acc exactly like `merge`, but for every row at once
            #         (hint: corr[:, None] * acc + p @ V[j:j+Bc]);  then m = m_new
            raise NotImplementedError("Fill in the tile loop")
        O[i:i + Br] = acc / l[:, None]
    return O

In [ ]:
n, d = 100, 16          # 100 is deliberately not a multiple of the block sizes
Q, K, V = (rng.normal(size=(n, d)) for _ in range(3))
ref = standard_attention(Q, K, V)
for Br, Bc in [(16, 16), (7, 32), (100, 1), (1, 100), (64, 64)]:
    out = flash_attention(Q, K, V, Br=Br, Bc=Bc)
    err = np.abs(out - ref).max()
    assert err < 1e-10, f"Br={Br}, Bc={Bc}: max error {err}"
print("✅ tiled attention matches standard attention; max |difference| =", np.abs(flash_attention(Q, K, V) - ref).max())

## 4. The causal mask, tile by tile
A decoder may only attend to keys $j \le i$. With tiles there are three cases:

* a tile entirely **in the past** of its query block: no masking needed;
* a tile on the **diagonal**: set the future scores to $-\infty$ inside the tile;
* a tile entirely **in the future**: every weight would be 0, so **skip it** (and every later tile).

Edit your tile loop (copy the cell above, or edit it in place) so that `causal=True` does this. Hints: `rows` holds the query positions;
the tile's key positions are `np.arange(j, j + s.shape[1])`; the whole tile is in the future when `j > rows[-1]`.

In [ ]:
ref_c = standard_attention(Q, K, V, causal=True)
for Br, Bc in [(16, 16), (7, 32), (32, 8)]:
    out = flash_attention(Q, K, V, Br=Br, Bc=Bc, causal=True)
    assert np.abs(out - ref_c).max() < 1e-10, f"causal Br={Br}, Bc={Bc} is off. Did you mask the diagonal tiles?"

# How many tiles does causal masking let you skip?
Br = Bc = 16
total = sum(1 for i in range(0, n, Br) for j in range(0, n, Bc))
used = sum(1 for i in range(0, n, Br) for j in range(0, n, Bc) if j <= min(i + Br, n) - 1)
print(f"✅ causal tiled attention matches. Tiles computed: {used} of {total} ({used / total:.0%})")

## 5. Experiment: how much memory does tiling save?
Standard attention's biggest temporary is the $n \times n$ score matrix. The tiled version's is one $B_r \times B_c$ tile plus a few length-$n$ vectors.
Run the cell to compare (float32, one head), then try changing the tile size.

In [ ]:
Br = Bc = 128
ns = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
std_mb = [n * n * 4 / 2**20 for n in ns]
tiled_mb = [(Br * Bc + 2 * n) * 4 / 2**20 for n in ns]   # one score tile + running m and l for every row
for n_, s_, t_ in zip(ns, std_mb, tiled_mb):
    print(f"n = {n_:6d}:  full S = {s_:10.1f} MiB   tiled temporaries = {t_:6.2f} MiB")

plt.figure(figsize=(5.6, 3.6))
plt.loglog(ns, std_mb, "o-", label="standard: n × n scores")
plt.loglog(ns, tiled_mb, "o-", label=f"tiled: {Br}×{Bc} tile + O(n) stats")
plt.xlabel("sequence length n"); plt.ylabel("MiB per head (float32)")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

**Think about it** (answers below the solutions):
1. Your tiled loop reads all of `K` and `V` once **per query block**. How many times is `K` read in total for $n = 4096$ and $B_r = 128$? Why does a bigger SRAM (bigger $B_r$) reduce HBM traffic?
2. The backward pass needs the probabilities $P$. FlashAttention doesn't store them. What does it save instead, and why is that enough?
3. In the merge step, why does starting from $m = -\infty$ work without a special case for the first block?

---
## Solutions
Only run these if you're stuck. They redefine the functions; re-run the check cells afterwards.

In [ ]:
def merge(state, scores, values):
    """One online-softmax update for a single query row.
    state  = (m, l, a): running max (float), running sum (float), running weighted sum of values (array of shape (d_v,))
    scores = (B,) scores of the new block,  values = (B, d_v) values of the new block.
    Returns the new (m, l, a)."""
    m, l, a = state
    m_new = max(m, float(np.max(scores)))
    corr = np.exp(m - m_new)                 # rescales the old terms; exp(-inf) = 0 on the first block
    e = np.exp(scores - m_new)               # new block's terms, relative to the new max
    return m_new, corr * l + e.sum(), corr * a + e @ values

In [ ]:
def online_softmax_row(scores, values, block=2):
    """softmax(scores) @ values for ONE query row, visiting the keys `block` at a time."""
    state = (-np.inf, 0.0, np.zeros(values.shape[1]))
    for start in range(0, len(scores), block):
        state = merge(state, scores[start:start + block], values[start:start + block])
    m, l, a = state
    return a / l

In [ ]:
def flash_attention(Q, K, V, Br=16, Bc=16, causal=False):
    """Tiled attention with an online softmax. Never builds an (n, m) matrix:
    the biggest temporary is one (Br, Bc) tile of scores."""
    n, d = Q.shape
    O = np.zeros((n, V.shape[1]))
    scale = 1.0 / np.sqrt(d)
    for i in range(0, n, Br):                       # one block of query rows
        q = Q[i:i + Br]
        rows = np.arange(i, i + len(q))
        m = np.full(len(q), -np.inf); l = np.zeros(len(q)); acc = np.zeros((len(q), V.shape[1]))
        for j in range(0, K.shape[0], Bc):          # stream the K/V blocks
            if causal and j > rows[-1]:
                break                               # every key in this block is in the future: skip it
            s = (q @ K[j:j + Bc].T) * scale          # (Br, Bc) tile of scores
            if causal:
                cols = np.arange(j, j + s.shape[1])
                s = np.where(cols[None, :] > rows[:, None], -np.inf, s)
            m_new = np.maximum(m, s.max(axis=1))
            corr = np.exp(m - m_new)
            p = np.exp(s - m_new[:, None])
            l = corr * l + p.sum(axis=1)
            acc = corr[:, None] * acc + p @ V[j:j + Bc]
            m = m_new
        O[i:i + Br] = acc / l[:, None]
    return O

**Think-about-it answers**
1. There are $4096 / 128 = 32$ query blocks, so `K` and `V` are each streamed 32 times. A bigger $B_r$ means fewer query blocks and fewer passes over `K`/`V`. That is where the paper's $\Theta(n^2 d^2 / M)$ HBM-access bound comes from: the number of passes shrinks as the SRAM size $M$ grows.
2. It saves the output $O$ and one number per row, the log-sum-exp $L_i = m_i + \log \ell_i$. In the backward pass it recomputes each score tile $S_{ij}$ from $Q$ and $K$ and gets $P_{ij} = \exp(S_{ij} - L_i)$ directly, tile by tile, so $P$ never has to be stored.
3. With $m = -\infty$ the correction factor is $e^{-\infty - m'} = 0$, which wipes out the (zero) old state, and the new block's terms are computed relative to its own max. One rule covers every block.

**Next:** M5.5, speculative decoding: another way to make generation faster without changing what the model outputs.